<a href="https://colab.research.google.com/github/Kavyasrimedi/flyrank-week1/blob/main/notebooks/02_your_first_readable_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kavyasrimedi/flyrank-week1/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

This notebook explores how to build a machine learning model that is not only effective but also **interpretable**. The main idea is that the 'model is just a rule you can read'. You'll be comparing a simple, hand-coded rule with a machine-learned decision tree to rank web pages and identify those that are 'declining'.

## 0. Setup (Colab or local)

This section sets up the environment and loads the necessary data. It checks if the notebook is running in Google Colab, clones a GitHub repository if needed, installs dependencies, and then loads the `content_refresh_anonymized.csv` dataset into a pandas DataFrame named `df`.

A crucial step here is the creation of the `is_declining_label`. This is the **target variable** for our prediction task. A page is labeled as 'declining' if its `trend_direction` is 'down'. This is converted into a binary integer (1 for declining, 0 otherwise). The code then prints the total number of pages and the rate at which pages are declining in the dataset.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

Here, you're creating a **hand-coded rule** based on intuition. The idea is that a page is 'worth reviewing' if it's been a while since its last update (**stale**) and it's still receiving a significant number of views (**visible**). The `hand_rule_score` is calculated by multiplying `stale` (1 if `>= 180` days, 0 otherwise) and `visible` (1 if `impressions_90d >= 500`, 0 otherwise) with the `impressions_90d`. This means only pages that are both stale and visible will get a non-zero score, and among those, higher impressions lead to a higher score.

The `top10` DataFrame then shows the top 10 pages ranked by this `hand_rule_score`, displaying relevant features like impressions, update age, average position, click-through rate (CTR), and trend direction.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

This section introduces a metric to evaluate the effectiveness of any ranking system: **Precision@K**. This metric measures, out of the top `K` pages identified by the ranking model, what fraction are *actually* 'declining' (based on our `is_declining_label`).

The `precision_at_k` function takes the scores generated by a ranking method, the true labels, and the `K` value. It sorts the pages by their scores, selects the top `K`, and then calculates the mean of their true labels (which are 0 or 1), effectively giving the proportion of truly declining pages among the top `K`.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


This code applies the `precision_at_k` function to the `hand_rule_score` calculated earlier. It calculates and prints the Precision@20 and Precision@50 for the hand-coded rule, giving us a baseline for how well our intuitive rule performs at identifying declining pages at different `K` values.

## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

Now, instead of manually crafting rules, you let a machine learning model learn them. The choice of a **depth-2 decision tree** is deliberate: its simplicity (only 3 yes/no questions) ensures that the learned rules are easily human-readable. This directly supports the notebook's theme of having a 'readable model'.

The features used for training (`content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`) are selected to be 'pre-decision signals,' meaning they are observable before the page's trend is known, preventing data leakage.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



This cell trains the `DecisionTreeClassifier` with `max_depth=2` on the selected features (`X`) and the `is_declining_label` (`y`). `class_weight="balanced"` helps handle potential class imbalance. The `random_state` ensures reproducibility.

The `export_text(tree, feature_names=features)` function is crucial here. It prints the trained decision tree in a text format, making its internal logic (the if/else rules it learned) directly visible and understandable. This is the 'readable rule' that the model found.

That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

This section explains that the previous text output *is* the model. Now you'll use this machine-learned model to rank pages and compare its performance to the hand-coded rule. The `predict_proba(X)[:, 1]` method gets the probability that each page belongs to class 1 (i.e., is declining) according to the decision tree.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


This cell calculates the Precision@K for the decision tree model's scores (`tree_score`) at K=20 and K=50. It then prints a comparison between the hand rule's Precision@K and the tree's Precision@K, allowing you to see which method performs better at identifying declining pages within the top ranked items.

Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

This markdown cell provides an important caveat about interpreting the results. It highlights that a depth-2 tree has limited scoring capabilities, leading to many tied scores. The exact winner (hand rule vs. tree) can depend on minor factors like library versions due to tie-breaking. The key takeaway is to describe *your specific run's results* honestly rather than making a general claim that one is always better. It also points out that while simple human rules can be strong at the very top, models might find deeper signals.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

This section demonstrates a critical concept in machine learning: **data leakage**. Leakage occurs when information about the target variable (the outcome you're trying to predict) is accidentally included in the features used for training. In this case, `trend_pct` is the raw percentage change from which the `trend_direction` (and thus `is_declining_label`) is derived. Therefore, including `trend_pct` as a feature is essentially giving the model the answer in advance.

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



This cell explicitly shows the effect of data leakage. A new feature set `X_leaky` is created by adding `trend_pct` to the original features. A new decision tree (`leaky`) is trained on this `X_leaky`. The resulting Precision@50 is `1.000`, which looks 'amazing' but is misleading. The printout of the 'leaky' tree clearly shows it just splits on `trend_pct`, effectively memorizing the label rather than learning a useful, generalizable rule. This illustrates why preventing leakage is crucial for building robust models.

The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

This cell summarizes the concept of leakage, reinforcing that when a feature is directly derived from or inherently contains the answer, it's leakage. It also emphasizes the importance of using only 'observable signals' that would be known *before* the outcome, a fundamental principle for building predictive models.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

This section provides prompts for your own experiments, encouraging you to interact with the code and explore how changes impact the model. Specifically, it suggests:

*   **Changing `max_depth`:** Experimenting with depths 3 or 4 to see if performance improves and if the tree remains readable.
*   **Swapping features:** Replacing `impressions_90d` with `engagement_rate` to observe how the tree's splits change.
*   **Important caveat (train/test split):** Reminding you that the current evaluation is 'in-sample' (training and testing on the same data), and a real-world scenario would require a proper train/test split (or 'client-holdout validation') to get a more reliable performance estimate.

In [11]:
# Your experiment here
from sklearn.tree import DecisionTreeClassifier, export_text

features1 = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X1 = df[features1].replace([np.inf, -np.inf], np.nan).fillna(0)

tree1 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42)
tree1.fit(X, y)

print(export_text(tree1, feature_names=features1))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- impressions_90d <= 73.50
|   |   |   |   |--- class: 1
|   |   |  

In this cell, you conducted an experiment by training a `DecisionTreeClassifier` with `max_depth=4` using the original features (`features1`). You then printed out the decision tree. As you can see, increasing the `max_depth` allows the tree to create more complex rules with more splits, which might capture more nuances in the data but also makes the tree harder to read compared to a depth-2 tree. This output shows a more detailed set of if/else conditions.

In [12]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "engagement_rate",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



In this cell, you performed another experiment as suggested in section 4. You replaced `impressions_90d` with `engagement_rate` in the `features` list and trained a new `DecisionTreeClassifier` with `max_depth=2`. The output `print(export_text(tree, feature_names=features))` shows what features the decision tree chose to split on first and how it constructed its rules based on this new feature set. In this case, it primarily splits on `avg_position` and then `content_age_days`.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.

This final section is a wrap-up, encouraging you to save your work and reiterating the core lessons of the notebook: **discover insights before modeling** (understand your data and create good features) and **prefer readable models that can't be easily fooled** (like shallow decision trees, and being vigilant about data leakage).